# First-diff logistic: grid search tuning

Grid search over `(window, low_thresh, high_thresh)`, visualise the AUROC surface, fit the final model, and save it.

In [ ]:
import sys
sys.path.append("../../..")

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

import models.first_diff_logistic as fdl

with open("../../../data_processing/dataset.pkl", "rb") as f:
    dataset = pickle.load(f)

rng = np.random.default_rng(0)
ev_ids   = [d for d, (has_car, *_) in dataset.items() if has_car]
noev_ids = [d for d, (has_car, *_) in dataset.items() if not has_car]

def split(ids, test_frac=0.3):
    n = max(1, int(len(ids) * test_frac))
    test = list(rng.choice(ids, n, replace=False))
    return [d for d in ids if d not in set(test)], test

ev_train, ev_test = split(ev_ids)
nev_train, nev_test = split(noev_ids)
train_homes = {d: dataset[d] for d in ev_train + nev_train}
test_homes  = {d: dataset[d] for d in ev_test  + nev_test}

print(f"Train: {len(train_homes)} homes ({len(ev_train)} EV)")
print(f"Test:  {len(test_homes)} homes ({len(ev_test)} EV)")

## Grid search — log full results

In [ ]:
loads = [df["load"].to_numpy() for _, (_, _, df) in train_homes.items()]
y     = np.array([int(has_car) for _, (has_car, _, _) in train_homes.items()])

windows   = [2, 4, 6, 8]
low_grid  = np.arange(0.4, 1.6, 0.2)
high_grid = np.arange(1.2, 3.2, 0.2)

records = []
for w in windows:
    for lo in low_grid:
        for hi in high_grid:
            if hi <= lo:
                continue
            X = [[fdl.transitions_per_day(load, w, lo, hi)] for load in loads]
            model = LogisticRegression(class_weight="balanced", max_iter=1000).fit(X, y)
            score = roc_auc_score(y, model.predict_proba(X)[:, 1])
            records.append({"window": w, "low": round(lo, 2), "high": round(hi, 2), "auroc": score})

grid_df = pd.DataFrame(records)
print(grid_df.sort_values("auroc", ascending=False).head(10))

## Visualise — AUROC heatmap per window size

In [ ]:
fig, axes = plt.subplots(1, len(windows), figsize=(14, 4), sharey=True)

for ax, w in zip(axes, windows):
    sub = grid_df[grid_df["window"] == w]
    pivot = sub.pivot(index="high", columns="low", values="auroc")
    im = ax.imshow(pivot.values, aspect="auto", origin="lower",
                   vmin=grid_df["auroc"].min(), vmax=1.0, cmap="viridis")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns.round(1), fontsize=7)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index.round(1), fontsize=7)
    ax.set_title(f"window={w} ({w*15} min)")
    ax.set_xlabel("low_thresh (kW)")
    if ax is axes[0]:
        ax.set_ylabel("high_thresh (kW)")

fig.colorbar(im, ax=axes, label="Train AUROC")
plt.suptitle("Grid search: train AUROC by (low_thresh, high_thresh)", y=1.02)
plt.tight_layout()
plt.show()

## Fit final model and evaluate on test homes

In [ ]:
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss

model, window, low, high = fdl.tune(train_homes)
results, charge_states = fdl.predict(model, test_homes, window, low, high)

y_test, p_test = results["has_ev"].to_numpy(), results["p_hat"].to_numpy()
print(f"AUROC    : {roc_auc_score(y_test, p_test):.4f}")
print(f"Log-loss : {log_loss(y_test, p_test):.4f}")
print(f"Brier    : {brier_score_loss(y_test, p_test):.4f}")
results.sort_values("p_hat", ascending=False)

## Save model

In [ ]:
fdl.save(model, window, low, high, "../../../models/first_diff_logistic.pkl")
print("Saved.")